# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [37]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
pip install python-dotenv


In [38]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [4]:
#!pip install pandas

In [39]:
import pandas as pd
df = pd.read_csv('Data.csv')

In [40]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\r\n,I loved this product. But they only seem to l...


## LLMChain

In [7]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.8 MB/s eta 0:00:00


In [8]:
pip install langchain-openai


In [9]:
!pip install -U langchain-openai
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [41]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=0.9)


In [42]:
prompt = ChatPromptTemplate.from_template(
    "Write a detailed and engaging product description for a {Product}. "
    "Highlight its key features, benefits, ideal use cases, and why it stands out from similar products. "
    "Use a persuasive and informative tone, suitable for an e-commerce website."
)



In [43]:
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.9)
prompt1 = PromptTemplate(template="Guess the name of the company that produced this {Product}.",
    input_variables=["Product"])
prompt1output=prompt1


chain = LLMChain(llm=llm, prompt=prompt1)
product = df['Product'][0]
chain.run(product)


'Grandeur Linens'

In [44]:
print(df['Product'][0])


Queen Size Sheet Set


In [45]:
product = df['Product'][0]
chain.run(product)

'There is not enough information provided to accurately guess the name of the company that produced the Queen Size Sheet Set.'

In [47]:
product = df['Product'][0]
chain.run(product)

'Royal Hotel Embroidered Queen Size Sheet Set.'

In [48]:
product = df['Product'][0]
chain.run(product)

'Dreaming Wapiti'

In [49]:
for index, row in df.iterrows():
    product_text = row["Product"]
    result = chain.run({"Product": product_text})  # استخدم نفس اسم المفتاح
    print(f"Product: {product_text}\nSummary: {result}\n")


Product: Queen Size Sheet Set
Summary: The company that produced this Queen Size Sheet Set is likely not specified in the text provided. Without more information, it is impossible to guess the name of the company.

Product: Waterproof Phone Pouch
Summary: It is difficult to guess the specific company without more information, but some popular companies known for producing waterproof phone pouches include Mpow, JOTO, and Vansky.

Product: Luxury Air Mattress
Summary: Simba

Product: Pillows Insert
Summary: MyPillow

Product: Milk Frother Handheld

Summary: Milk Boss

Product: L'Or Espresso Café 

Summary: Nestlé

Product: Hervidor de Agua Eléctrico
Summary: The company that produced this Electric Water Kettle is likely called "Oster."



## SimpleSequentialChain

In [52]:
from langchain.chains import SimpleSequentialChain

In [53]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    # This prompt guesses the company name based on the product name
    "Guess the name of the company that produced the following product:\n\n{Product}"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [54]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    # This prompt gives a professional description of the company based on its name
    "Provide a detailed and engaging description of the company called {company_name}. "
    "Include what the company is known for, its industry, and any notable achievements."
)

# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)


In [55]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [56]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
The company that produced the Queen Size Sheet Set could be a variety of different bedding and home goods companies, such as Pottery Barn, Brooklinen, Target, or AmazonBasics. Without more specific information, it is impossible to determine the exact company.
One notable company that produces high-quality bedding and home goods is Brooklinen. Founded in 2014, Brooklinen has become a household name known for its luxurious and affordable products. Specializing in bedding sets, including the popular Queen Size Sheet Set, Brooklinen is dedicated to providing customers with the ultimate comfort and style for their homes.

Brooklinen prides itself on using only the finest materials, such as long-staple cotton and ethically-sourced materials, to ensure that their products are not only comfortable but also sustainable. Their Queen Size Sheet Set, with its soft and smooth fabric, is designed to provide a luxurious sleep experience for customers.



'One notable company that produces high-quality bedding and home goods is Brooklinen. Founded in 2014, Brooklinen has become a household name known for its luxurious and affordable products. Specializing in bedding sets, including the popular Queen Size Sheet Set, Brooklinen is dedicated to providing customers with the ultimate comfort and style for their homes.\n\nBrooklinen prides itself on using only the finest materials, such as long-staple cotton and ethically-sourced materials, to ensure that their products are not only comfortable but also sustainable. Their Queen Size Sheet Set, with its soft and smooth fabric, is designed to provide a luxurious sleep experience for customers.\n\nIn addition to their high-quality products, Brooklinen has received numerous accolades and achievements in the industry. They have been featured in top publications such as Vogue, Forbes, and The New York Times, and have garnered a loyal following of customers who swear by their products.\n\nOverall, B

**Repeat the above twice for different products**

## SequentialChain

In [57]:
from langchain.chains import SequentialChain

In [58]:
llm = ChatOpenAI(temperature=0.9)


first_prompt = ChatPromptTemplate.from_template(
  "Translate the following product review into English:\n\n{review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="translated_review"
                    )


In [59]:
second_prompt = ChatPromptTemplate.from_template(
    # This prompt summarizes a product review in a concise and informative way
    "Summarize the following product review:\n\n{translated_review}"
)

chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    output_key="summary")

In [60]:
# prompt template 3: translate to English or another language
third_prompt = ChatPromptTemplate.from_template(
    # This prompt detects the language of the review and translates it into the target language
    "Detect the language of the following product review, then translate it into {target_language}:\n\n{review}"
)

# chain 3: input = review, target_language / output = translated_review
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    output_key="translated_review"
)

In [61]:
# prompt template 4: follow up message that takes as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
        # This prompt generates a follow-up message based on the translated review and its summary
        "Based on the following translated review:\n\n{translated_review}\n\n"
        "And this summary:\n\n{summary}\n\n"
        "Write a polite and professional follow-up message to the customer."
)

chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="follow_up_message"  # give a name to this output
                     )


In [62]:
# overall_chain: input= Review
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_three, chain_two, chain_four],
    input_variables=["review", "target_language"],
    output_variables=["translated_review", "summary", "follow_up_message"],
    verbose=True
)



In [63]:
review = df.Review[5]
# overall_chain(review)

overall_chain({
    "review": df.Review[5],
    "target_language": "English"
})



> Entering new SequentialChain chain...

> Finished chain.


{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'target_language': 'English',
 'translated_review': "Language: French\n\nTranslation: I find the taste mediocre. The foam doesn't hold, it's strange. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?",
 'summary': 'The reviewer finds the taste of the product mediocre and notes that the foam does not hold well. They suspect that the product may be from an old batch or could be counterfeit, as they have purchased the same product in stores with a much better taste.',
 'follow_up_message': 'Dear Valued Customer,\n\nThank you for taking the time to provide feedback on your recent purchase. We apologize that the taste of the product did not meet your expectations and that the foam did not hold well for you.\n\nWe take quality control very seriously and will investigate wheth

**Repeat the above twice for different products or reviews**

## Router Chain

In [64]:
creative_career_template = """You are a career advisor who specializes in creative fields. \
You help people turn their artistic talents—like design, writing, photography, or content creation—into fulfilling careers. \
You provide inspiring and practical advice to turn passion into profession.

Here is a question:
{input}"""

freelance_career_template = """You are a freelancing career coach. \
You guide people who want to work independently—helping them build portfolios, find clients, price their services, and grow sustainably. \
You provide clear, actionable advice for launching and succeeding as a freelancer.

Here is a question:
{input}"""

tech_career_template = """You are a professional tech career strategist. \
You guide people toward roles in software engineering, AI, data science, cybersecurity, and more. \
You suggest learning paths, in-demand skills, and job-hunting strategies in the tech world.

Here is a question:
{input}"""

business_career_template = """You are a business career expert. \
You help people explore jobs in marketing, finance, management, and entrepreneurship. \
You offer smart guidance on building a successful career in the business world.

Here is a question:
{input}"""


In [65]:
prompt_infos = [
    {
        "name": "creative",
        "description": "Helps people explore careers in creative fields like design, writing, and content creation.",
        "prompt_template": creative_career_template
    },
    {
        "name": "freelance",
        "description": "Provides career advice for individuals interested in freelancing and remote work.",
        "prompt_template": freelance_career_template
    },
    {
        "name": "tech",
        "description": "Guides users toward careers in tech, programming, AI, and cybersecurity.",
        "prompt_template": tech_career_template
    },
    {
        "name": "business",
        "description": "Advises on careers in marketing, management, finance, and entrepreneurship.",
        "prompt_template": business_career_template
    }
]


In [66]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [67]:
llm = ChatOpenAI(temperature=0)

In [68]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [69]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [71]:
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE


In [72]:
destinations_str = "\n".join(
    [f"{p['name']}: {p['description']}" for p in prompt_infos]
)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)


In [73]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [74]:
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

In [75]:
chain.run("I’m passionate about space and physics. What careers should I explore?")



> Entering new MultiPromptChain chain...
tech: {'input': 'I’m passionate about space and physics. What careers should I explore?'}
> Finished chain.


'If you are passionate about space and physics, there are several exciting career paths in the tech industry that you may want to explore:\n\n1. Aerospace Engineering: Aerospace engineers design, develop, and test aircraft, spacecraft, and missiles. They work on projects related to space exploration, satellite technology, and more.\n\n2. Astrophysics: Astrophysicists study the physical properties and processes of celestial bodies and the universe as a whole. They often use advanced computational tools and data analysis techniques to conduct research.\n\n3. Data Science in Space Exploration: Data scientists play a crucial role in analyzing large datasets collected from space missions and telescopes. They use machine learning and statistical techniques to extract valuable insights from the data.\n\n4. Space Technology: Space technology encompasses a wide range of roles, including satellite engineering, space systems design, and space mission planning. Professionals in this field work on 

In [76]:
chain.run("I enjoy solving math problems. What kind of job fits my skills?")



> Entering new MultiPromptChain chain...
tech: {'input': 'I enjoy solving math problems. What kind of job fits my skills?'}
> Finished chain.


'With your enjoyment of solving math problems, you may be well-suited for a career in data science or machine learning. These fields involve using mathematical and statistical techniques to analyze data, make predictions, and derive insights. \n\nTo pursue a career in data science or machine learning, you may want to focus on learning programming languages such as Python or R, as well as gaining proficiency in data manipulation, statistical analysis, and machine learning algorithms. Additionally, building a strong foundation in linear algebra, calculus, and probability theory can be beneficial for these roles.\n\nJob titles to consider in this field include data scientist, machine learning engineer, or quantitative analyst. These roles often involve working with large datasets, developing predictive models, and using algorithms to extract valuable information from data.\n\nOverall, a career in data science or machine learning could be a great fit for someone who enjoys solving math pro

In [77]:
chain.run("I love biology and understanding how life works. What are some career options for me?")



> Entering new MultiPromptChain chain...
creative: {'input': 'I love biology and understanding how life works. What are some career options for me in creative fields like design, writing, and content creation?'}
> Finished chain.


"That's great to hear that you have a passion for biology! There are actually several creative career options that combine your love for science with design, writing, and content creation. Here are a few ideas for you to consider:\n\n1. Scientific Illustrator: As a scientific illustrator, you can use your artistic skills to create visual representations of biological concepts, organisms, and processes. This can include creating illustrations for textbooks, scientific journals, educational materials, and even museum exhibits.\n\n2. Science Writer: If you have a talent for writing, you could pursue a career as a science writer. You could write articles, blog posts, or even books about biology-related topics, making complex scientific concepts accessible and engaging for a general audience.\n\n3. Content Creator for Science Communication: With the rise of social media and online platforms, there is a growing demand for content creators who can communicate scientific information in a creat

**Repeat the above at least once for different inputs and chains executions - Be creative!**